# Tugas Akhir Praktikum Logical Agents: Wumpus World 5×5

## Identitas
- **Nama**   : Wayan Raditya Putra  
- **NRP**    : 5054241029  
- **Departemen** : Teknik Informatika  
- **Prodi** : RKA
- **Mata Kuliah** : Kecerdasan Komputasional
- **Dosen Pengampu** :  
  - Prof. Dr. Eng. Nanik Suciati, S.Kom., M.Kom.  
  - Imam Mustafa Kamal, S.ST, Ph.D.  

---

## Deskripsi Tugas
Agen ditempatkan pada lingkungan **Wumpus World 5×5**.  
- Posisi awal agen: `[1,1]`  
- Posisi emas: `[2,4]`  
- Posisi pit: `(1,3), (1,5), (4,1), (4,5)`  
- Posisi Wumpus: `(3,1)`  

Aturan Wumpus World:  
- Pit menimbulkan **breeze** di sel tetangga (atas, bawah, kiri, kanan).  
- Wumpus menimbulkan **stench** di sel tetangga (atas, bawah, kiri, kanan).  

---

## Tujuan
1. Mendefinisikan proposisi dan aturan logika (R1–Rn) berdasarkan aturan Wumpus World.  
2. Menyusun rangkaian proposisi secara sistematis dari posisi awal hingga emas.  
3. Melakukan inferensi menggunakan **Truth Table Entailment (TT-entails)** untuk memverifikasi kebenaran inferensi.  
4. Melakukan inferensi menggunakan **Forward Chaining** (NRP ganjil).  
5. Membuktikan apakah agen dapat mencapai emas dengan aman, serta menuliskan jalur langkah demi langkah.  
6. Mengimplementasikan program Python yang merepresentasikan inferensi logika agen.  
---

## Pendahuluan

Pada bidang **Kecerdasan Buatan (Artificial Intelligence)**, salah satu pendekatan yang digunakan untuk 
merepresentasikan pengetahuan adalah melalui **Logical Agents**. Logical agent merupakan agen yang 
mengambil keputusan berdasarkan **aturan logika** dan inferensi, bukan sekadar trial-and-error.  
Agen jenis ini cocok untuk lingkungan yang penuh ketidakpastian tetapi memiliki aturan formal yang 
jelas, seperti **Wumpus World**.

**Wumpus World** adalah dunia berbentuk grid yang dipopulerkan dalam literatur AI (Russell & Norvig). 
Lingkungan ini digunakan untuk menguji kemampuan agen dalam bernavigasi secara aman untuk mencapai 
tujuan (misalnya menemukan emas), sambil menghindari bahaya berupa **pit** (lubang) dan **wumpus** 
(monster). Agen hanya bisa merasakan indikator lingkungan:  
- **Breeze** → muncul di sel yang berdekatan dengan pit.  
- **Stench** → muncul di sel yang berdekatan dengan wumpus.  

Untuk dapat bernavigasi dengan aman, agen harus mampu:  
1. Mendefinisikan proposisi yang merepresentasikan kondisi dunia.  
2. Menggunakan aturan logika (R1–Rn) untuk melakukan inferensi.  
3. Menentukan langkah berikutnya berdasarkan inferensi yang sahih.

Dalam tugas ini digunakan dua metode inferensi utama:  
- **Truth Table Entailment (TT-entails)**: metode dasar dengan membangun tabel kebenaran untuk 
  memverifikasi apakah suatu proposisi logis benar secara konsisten.  
- **Forward Chaining**: metode berbasis aturan produksi yang mulai dari fakta yang diketahui 
  kemudian menurunkan fakta-fakta baru sampai mencapai kesimpulan.  

Dengan pendekatan ini, agen diharapkan dapat membuktikan apakah emas di koordinat `(2,4)` dapat 
dicapai dari posisi awal `(1,1)` dengan aman sesuai aturan Wumpus World.


## Definisi Proposisi & Aturan Umum (R1–Rn)

### Proposisi
1. **P(x,y)** : Ada *pit* pada sel `(x,y)`  
2. **W(x,y)** : Ada *wumpus* pada sel `(x,y)`  
3. **G(x,y)** : Ada emas (*gold*) pada sel `(x,y)`  
4. **B(x,y)** : Ada *breeze* pada sel `(x,y)`  
5. **S(x,y)** : Ada *stench* pada sel `(x,y)`  
6. **OK(x,y)** : Sel `(x,y)` aman untuk dimasuki agen  
7. **¬OK(x,y)** : Sel `(x,y)` berbahaya (ada kemungkinan pit atau wumpus) 

---

### Aturan Umum (R1–Rn)
**R1 (Breeze Rule)**  
B(x,y) ↔ (P(x-1,y) ∨ P(x+1,y) ∨ P(x,y-1) ∨ P(x,y+1))

**R2 (Stench Rule)**  
S(x,y) ↔ (W(x-1,y) ∨ W(x+1,y) ∨ W(x,y-1) ∨ W(x,y+1))
  
**R3 (Safety Rule)**  
  ¬B(x,y) ∧ ¬S(x,y) → OK(x,y)
  
**R4 (Pit Inference)**  
Jika ada breeze di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti pit.  

**Notasi:**
B(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → P(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  

**R5 (Wumpus Inference)**  
Jika ada stench di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti wumpus.  

**Notasi:**
S(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → W(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  


**R6 (Gold Detection)**  
Jika ada emas di `(x,y)`, maka `(x,y)` adalah tujuan agen.  

**Notasi:**
G(x,y) → Goal(x,y)

## Project Setup

Pada tahap ini dilakukan proses **import library** dan modul pendukung yang dibutuhkan untuk mengerjakan tugas Logical Agents pada Wumpus World.  
Library yang digunakan terdiri dari modul internal (`utils.py`, `logic.py`, `agents.py`) maupun library eksternal Python.

### Penjelasan Library

* **utils** → berisi fungsi utilitas tambahan yang mendukung proses inferensi/logika.
* **logic** → berisi implementasi representasi logika (proposisi, aturan, inferensi).
* **agents** → modul berisi definisi agen dan cara agen berinteraksi dengan Wumpus World.
* **math** → library standar Python untuk operasi matematika.
* **inspect.getsource** → digunakan untuk menampilkan kode sumber dari fungsi tertentu (berguna saat analisis).
* **IPython.display.HTML** → menampilkan output HTML di Jupyter Notebook.
* **tabulate** → menghasilkan tabel rapi untuk menyajikan hasil inferensi dan analisis.



In [7]:
from utils import *
from logic import *
import agents
import math
from inspect import getsource
from IPython.display import HTML
from tabulate import tabulate

## Representasi Peta dan Informasi Wumpus World

Kelas `WumpusAgent` digunakan sebagai **alat bantu** untuk membangun representasi dunia Wumpus 
dalam bentuk grid (map) serta memasukkan informasi lingkungan ke dalam *knowledge base* (KB) agen.

- **World (Peta 5×5)** direpresentasikan sebagai list of list, di mana setiap cell dapat berisi:
  - `"P"` → Pit
  - `"W"` → Wumpus
  - `"G"` → Gold
  - `"B"` → Breeze
  - `"S"` → Stench
  - `[]`   → Kosong (tidak ada percept)

- **Knowledge Base (KB)** dibangun secara bertahap:
  1. Agen memulai dari posisi awal `(1,1)` → otomatis dianggap aman (`Safe(1,1)`).
  2. Agen membaca **percepts** di cell tersebut (misalnya Breeze, Stench).
  3. Informasi percepts dimasukkan ke dalam KB sebagai proposisi logika, contohnya:
     - `Breeze(1,2)`
     - `Stench(2,1)`
     - `Gold(2,4)`

Dengan cara ini, agen dapat menghubungkan **peta (map)** dan **pengetahuan logika** yang 
dibutuhkan untuk melakukan inferensi, sehingga jalur menuju emas dapat ditentukan secara aman.


In [10]:

# Sumber = Aima Data
class WumpusAgent:
    def __init__(self, world, initial_pos=(1,1)):
        self.world = world
        self.kb = PropKB()
        self.visited = set()
        self.position = initial_pos

        # Cell Agent pertama kali
        self.kb.tell(expr(f"Safe({initial_pos[0]},{initial_pos[1]})"))
        self.visited.add(initial_pos)

        # Update KB berdasarkan percepts
        self.update_kb(*initial_pos)

    def get_percepts(self, x, y):
        """
        Ambil percept dari koordinat (x,y) sesuai world
        """
        world_x = len(self.world) - y
        world_y = x - 1
        if 0 <= world_x < len(self.world) and 0 <= world_y < len(self.world[0]):
            return set(self.world[world_x][world_y])
        return set()

    def update_kb(self, x, y):
        """
        Update KB dengan percepts dari posisi (x,y). ini contohnya:
        """
        percepts = self.get_percepts(x, y)
        for p in percepts:
            self.kb.tell(expr(f"{p}({x},{y})"))  


In [16]:
world = [
    # y = 5
    [["P"], ["B"], ["B"], ["P"], ["B"]],   # row 5
    # y = 4
    [["B"], ["G"], [], ["B"], []],      # row 4
    # y = 3
    [["P"], [], [], [], []],            # row 3
    # y = 2
    [["B"], [], ["S"], ["B"], []],         # row 2
    # y = 1
    [[], ["S"], ["W"], ["P"], ["B"]]     # row 1
]

agent = WumpusAgent(world, initial_pos=(1,1))
